# Lab 5 · Phân khúc giá & host chuyên nghiệp (groupby, transform, merge)

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · Bài 5**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo bài 5 phân tích theo **quận**. Lab này phân tích thị trường theo hai chiều khác:
**phân khúc giá** (rẻ / trung / cao) và **kiểu host** (cá nhân / chuyên nghiệp), đồng thời thực hành
`map` → `groupby`/`agg` → `transform` → `pivot_table` → `merge`.

**Mục tiêu bài lab**

1. Phân biệt `loc` (nhãn) với `iloc` (vị trí) và hiểu phép toán **tự khớp theo index**.
2. Dùng `map` với **hàm** để sinh cột phân loại từ cột số.
3. Dùng `groupby` + `agg` đặt tên trên nhiều cột và diễn giải kết quả.
4. Dùng `transform` để gắn thông tin nhóm về từng dòng.
5. Tạo `pivot_table` hai chiều và `merge` bảng tra cứu có `validate`.

Phiên bản đề: `pandas-groupby-merge-v1` (24/09/2026). Lưu đúng tên **`lab-05.ipynb`**.

## Cách làm và nộp bài

- Phần khởi động (Bài 1–2) và các bài có hướng dẫn: nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 ở giờ lý thuyết đánh giá các kỹ năng này. Bài tự làm cuối lab được gắn nhãn ✅ mở:
  được dùng AI, kèm trách nhiệm khai báo theo chính sách AI của môn.
- Ghi dự đoán trước khi chạy; điền thân hàm TODO và phần trả lời Markdown.
- Giữ tên hàm, tham số, cell ID và metadata; viết helper/import thêm trong chính cell bài làm.
  Grader chỉ chạy cell câu đó với input riêng, không phụ thuộc biến `df`, `DEMO`, `phi` ở cell khác.
  `pd` (pandas) và `np` (NumPy) được cung cấp sẵn khi chấm.
- Public checks chỉ là ví dụ tự kiểm tra trên bảng giả lập nhỏ; test ẩn dùng bảng khác cùng hợp đồng.
  Một check lỗi không dừng các check sau. TODO báo **CHƯA LÀM**; lỗi cú pháp vẫn cần sửa.
- **Không sửa DataFrame/Series đầu vào** (không thêm cột như `df["chuyen"] = ...` bên trong hàm);
  không ghi cứng kết quả. Không cần xử lý input ngoài hợp đồng.
- Lab chạy với hành vi **Copy-on-Write** của pandas 3 (bật sẵn trong cell dữ liệu và khi chấm).
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.
- **Restart & Run all**, lưu notebook, commit/push vào repo của bạn trong org và tạo tag theo cuối bài.
  Notebook chạy hết khi còn TODO chưa có nghĩa đạt bài.

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Bài 1–8, mỗi bài 10 điểm | 80 | Tự động; giảng viên kiểm tra phương pháp |
| Dự đoán và diễn giải bảng (Bài 2, 4, 5, 7, 8) | 15 | Giảng viên |
| Khai báo AI và bằng chứng kiểm chứng | 5 | Giảng viên |

Yêu cầu dùng `map`, `agg` đặt tên, `transform`, `pivot_table`, `merge(validate=...)` được giảng viên
kiểm tra cùng code, không suy ra chỉ từ kết quả. Bài tự làm ✅ mở không cộng vào 80 điểm tự động.

In [ ]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")

## Dữ liệu dùng trong lab

Mặc định notebook mở bằng **12 dòng giả lập** (`DEMO`, giá CLP/đêm) để Run all được cả khi không có mạng.
Không dùng kết quả của 12 dòng này để kết luận về Santiago.

Khi thực hành trên dữ liệu thật, đặt `LISTINGS_PATH = SNAPSHOT_URL` — file `listings.csv` bản **visualisations**,
Santiago, snapshot **2026-06-29**, [Inside Airbnb](https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations/listings.csv) —
hoặc bản sao cùng snapshot do giảng viên cung cấp.

Cột dùng trong lab: `id`, `neighbourhood`, `price`, `number_of_reviews_ltm` (số review trong 12 tháng qua,
chỉ báo gần đúng — proxy — cho "mức độ được đặt") và `calculated_host_listings_count`
(host của phòng đang có bao nhiêu listing). Public checks luôn dùng `DEMO`.

In [ ]:
import numpy as np
import pandas as pd

# Hành vi pandas 3 (Copy-on-Write).
if pd.__version__.startswith("2."):
    pd.set_option("mode.copy_on_write", True)

SNAPSHOT_URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
                "2026-06-29/visualisations/listings.csv")

# 12 dòng GIẢ LẬP (CLP/đêm) — dùng cho public checks và khi chưa có snapshot thật.
DEMO = pd.DataFrame({
    "id": list(range(201, 213)),
    "neighbourhood": ["Santiago"] * 5 + ["Providencia"] * 4 + ["Ñuñoa"] * 2 + ["Vitacura"],
    "price": [25000.0, 45000.0, 120000.0, np.nan, 60000.0,
              30000.0, 99999.0, 100000.0, 18000.0,
              55000.0, 150000.0, 210000.0],
    "number_of_reviews_ltm": [10, 20, 3, 0, 15, 8, 12, 6, 2, 30, 1, 4],
    "calculated_host_listings_count": [1, 6, 12, 2, 5, 1, 1, 7, 3, 9, 1, 20],
})

# Đặt LISTINGS_PATH = SNAPSHOT_URL (hoặc đường dẫn file) để làm trên Santiago thật.
LISTINGS_PATH = None

if LISTINGS_PATH is None:
    df = DEMO.copy()
    DATA_LABEL = "GIẢ LẬP — 12 dòng DEMO, không phải Santiago"
else:
    df = pd.read_csv(LISTINGS_PATH)
    DATA_LABEL = f"Dữ liệu người học cung cấp: {LISTINGS_PATH}"

print(DATA_LABEL)
print("Số dòng:", len(df))

## Bài 1 · Khởi động: `loc` theo nhãn, `iloc` theo vị trí

```python
s = pd.Series([100, 200, 300], index=[2, 0, 5])
```

Dự đoán `s.loc[2]`, `s.iloc[2]`, `s.loc[0]`, `s.iloc[0]` trước khi chạy.

### Hợp đồng hàm · 10 điểm tự động

`loc_iloc(s: pd.Series, nhan, vi_tri: int) -> dict`

Nhận Series có index không trùng, một nhãn `nhan` có trong index và một vị trí `vi_tri` hợp lệ
(có thể âm như `-1`). Trả dict có đúng các khóa `theo_nhan` = phần tử mang **nhãn** `nhan`,
`theo_vi_tri` = phần tử ở **vị trí** `vi_tri`. Dùng `.loc` và `.iloc`, không dùng `s[...]` trực tiếp.

In [ ]:
def loc_iloc(s: pd.Series, nhan, vi_tri: int) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q1():
    s = pd.Series([100, 200, 300], index=[2, 0, 5])
    result = loc_iloc(s, 2, 2)
    assert set(result) == {"theo_nhan", "theo_vi_tri"}
    assert result["theo_nhan"] == 100, "loc[2] lấy phần tử có NHÃN 2"
    assert result["theo_vi_tri"] == 300, "iloc[2] lấy phần tử ở VỊ TRÍ thứ ba"
    assert loc_iloc(s, 0, 0) == {"theo_nhan": 200, "theo_vi_tri": 100}

public_check("Q1 · ví dụ công khai", check_q1)

## Bài 2 · Alignment: phép toán tự khớp theo index

```python
gia_cu = pd.Series({"A": 100, "B": 200, "C": 50})
gia_moi = pd.Series({"B": 220, "A": 90, "D": 70})
```

Dự đoán `gia_moi - gia_cu` trước khi chạy: phần tử nào được trừ với nhau, nhãn nào ra NaN?

### Hợp đồng hàm · 10 điểm tự động

`align_change(gia_cu: pd.Series, gia_moi: pd.Series) -> dict`

Nhận hai Series số, index là nhãn không trùng. Trả dict có đúng các khóa:
`thay_doi` = Series `gia_moi - gia_cu` (pandas tự khớp nhãn, index là hợp của hai index)
và `so_nan` = int, số NaN trong `thay_doi`. Không dùng vòng lặp, không sửa input.

In [ ]:
def align_change(gia_cu: pd.Series, gia_moi: pd.Series) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q2():
    gia_cu = pd.Series({"A": 100, "B": 200, "C": 50})
    gia_moi = pd.Series({"B": 220, "A": 90, "D": 70})
    result = align_change(gia_cu, gia_moi)
    assert set(result) == {"thay_doi", "so_nan"}
    thay_doi = result["thay_doi"]
    assert isinstance(thay_doi, pd.Series) and set(thay_doi.index) == {"A", "B", "C", "D"}
    assert thay_doi["A"] == -10 and thay_doi["B"] == 20
    assert result["so_nan"] == 2

public_check("Q2 · ví dụ công khai", check_q2)

**Giải thích:** A được trừ với … vì …; C và D thành NaN vì …

## Phần 1 · Phân tích thị trường theo hai chiều

## Bài 3 · Sinh cột phân khúc bằng `map` với hàm

Chia giá thành 3 phân khúc: `"re"` (< 30.000), `"trung"` (30.000 đến dưới 100.000), `"cao"` (≥ 100.000).
Bài này viết bằng `map` với hàm để luyện cách sinh cột phân loại; với phân khúc theo ngưỡng thì
`pd.cut` (đã học ở Bài 5) vừa vector hoá vừa nói rõ ý định hơn một chuỗi `if/elif`.

### Hợp đồng hàm · 10 điểm tự động

`phan_khuc(gia) -> str | None`

Nhận **một** giá: số hữu hạn không âm, `NaN` hoặc `None`. Trả `"re"`, `"trung"`, `"cao"` theo ngưỡng trên
(đúng 30.000 là `"trung"`, đúng 100.000 là `"cao"`); giá thiếu trả `None` — kiểm tra bằng `pd.isna` **trước**.
Hàm được dùng qua `df["price"].map(phan_khuc)`.
Đối chiếu snapshot: trung **12.325**, cao **3.703**, rẻ **1.660** dòng (còn lại thiếu giá).

In [ ]:
def phan_khuc(gia):
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q3():
    assert phan_khuc(29999.0) == "re" and phan_khuc(30000.0) == "trung"
    assert phan_khuc(99999.0) == "trung" and phan_khuc(100000.0) == "cao"
    assert phan_khuc(np.nan) is None and phan_khuc(None) is None
    dem = DEMO["price"].map(phan_khuc).value_counts(dropna=False)
    assert dem["trung"] == 5 and dem["cao"] == 4 and dem["re"] == 2

public_check("Q3 · ví dụ công khai", check_q3)

In [ ]:
def them_phan_khuc():
    df["phan_khuc"] = df["price"].map(phan_khuc)
    df["chuyen"] = df["calculated_host_listings_count"] >= 5   # quy ước ở Bài 5
    return df["phan_khuc"].value_counts(dropna=False)

# Các cell thực hành sau dùng hai cột này trên dữ liệu đang dùng.
preview("Bài 3 trên dữ liệu đang dùng", them_phan_khuc)

## Bài 4 · `agg` đặt tên — thống kê ba phân khúc

`number_of_reviews_ltm` là số review trong 12 tháng qua, được dùng làm chỉ báo gần đúng (proxy) cho "mức độ được đặt".

### Hợp đồng hàm · 10 điểm tự động

`segment_stats(df: pd.DataFrame) -> pd.DataFrame`

Nhận bảng có cột `id`, `phan_khuc` (chuỗi hoặc thiếu), `price`, `number_of_reviews_ltm`.
`groupby("phan_khuc")` rồi `agg` với ba cột đặt tên, đúng thứ tự:

- `so_phong = ("id", "size")` · `gia_trung_vi = ("price", "median")` · `review_nam_tb = ("number_of_reviews_ltm", "mean")`.

Index kết quả là các phân khúc có mặt (dòng thiếu phân khúc bị bỏ). Không làm tròn, không sửa input.
Đối chiếu snapshot: trung có **12.325** phòng; trung vị giá phân khúc cao **145.318**; review TB phân khúc trung ≈ **15,4**.

In [ ]:
def segment_stats(df: pd.DataFrame) -> pd.DataFrame:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q4():
    demo = DEMO.assign(phan_khuc=["re", "trung", "cao", None, "trung", "trung",
                                  "trung", "cao", "re", "trung", "cao", "cao"])
    tk = segment_stats(demo)
    assert isinstance(tk, pd.DataFrame)
    assert list(tk.columns) == ["so_phong", "gia_trung_vi", "review_nam_tb"]
    assert set(tk.index) == {"re", "trung", "cao"}
    assert tk.loc["trung", "so_phong"] == 5 and tk.loc["cao", "gia_trung_vi"] == 135000.0
    assert tk.loc["trung", "review_nam_tb"] == 17.0

public_check("Q4 · ví dụ công khai", check_q4)

In [ ]:
preview("Bài 4 trên dữ liệu đang dùng", lambda: segment_stats(df).round(1))

**Đọc bảng** (ghi rõ dữ liệu giả lập hay Santiago): phân khúc nào nhiều phòng nhất? Review trung bình thay đổi thế nào theo giá? …

## Bài 5 · Host chuyên nghiệp

Cột `calculated_host_listings_count` cho biết host của phòng đang có bao nhiêu listing.
Quy ước host có **≥ 5 listing** là "chuyên nghiệp".

### Hợp đồng hàm · 10 điểm tự động

`pro_host_summary(df: pd.DataFrame, nguong: int = 5) -> dict`

Nhận bảng có cột `calculated_host_listings_count`, `price`, `number_of_reviews_ltm` và ít nhất một dòng.
Trả dict có đúng các khóa:

- `chuyen`: Series bool `calculated_host_listings_count >= nguong`, cùng index với `df`;
- `ty_le_chuyen`: float, tỷ lệ dòng thuộc host chuyên nghiệp;
- `so_sanh`: DataFrame `groupby` theo Series `chuyen`, index là các giá trị `False`/`True` có mặt,
  hai cột đặt tên đúng thứ tự `gia_trung_vi = ("price", "median")`, `review_nam_tb = ("number_of_reviews_ltm", "mean")`.

**Không thêm cột vào `df`**: nhóm trực tiếp bằng Series, ví dụ `df.groupby(chuyen)`.
Đối chiếu snapshot: tỷ lệ **0,358**; trung vị giá nhóm chuyên nghiệp **65.078**; review TB nhóm cá nhân ≈ **12,7**.

In [ ]:
def pro_host_summary(df: pd.DataFrame, nguong: int = 5) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q5():
    before = DEMO.copy()
    result = pro_host_summary(DEMO)
    assert set(result) == {"chuyen", "ty_le_chuyen", "so_sanh"}
    assert isinstance(result["chuyen"], pd.Series) and result["chuyen"].dtype == bool
    assert result["ty_le_chuyen"] == 0.5
    so_sanh = result["so_sanh"]
    assert list(so_sanh.columns) == ["gia_trung_vi", "review_nam_tb"]
    assert so_sanh.loc[True, "gia_trung_vi"] == 80000.0 and so_sanh.loc[False, "review_nam_tb"] == 5.5
    pd.testing.assert_frame_equal(DEMO, before, obj="DEMO (không được thêm cột)")

public_check("Q5 · ví dụ công khai", check_q5)

In [ ]:
preview("Bài 5 trên dữ liệu đang dùng", lambda: pro_host_summary(df)["so_sanh"].round(1))

**Diễn giải:** nhóm chuyên nghiệp chiếm … và có giá/review … Các số liệu này đã đủ để kết luận host
chuyên nghiệp hoạt động hiệu quả hơn chưa, hay còn biến ẩn? (gợi ý: họ tập trung ở quận nào?) …

## Bài 6 · `transform` — gắn quy mô quận vào từng dòng

`transform` trả kết quả **đúng độ dài bảng gốc**. Dùng nó để gắn quy mô quận vào mỗi dòng,
sau đó lọc các quận quá nhỏ trước khi so sánh.

### Hợp đồng hàm · 10 điểm tự động

`big_districts(df: pd.DataFrame, nguong: int = 300) -> dict`

Nhận bảng có cột `neighbourhood` (chuỗi, không thiếu) và `id`. Trả dict có đúng các khóa:

- `n_quan`: Series số dòng của quận chứa mỗi dòng, lấy bằng `groupby("neighbourhood")[...].transform("size")`,
  cùng index và thứ tự với `df`;
- `df_dong`: DataFrame các dòng thuộc quận có **≥ nguong** dòng, giữ đúng các cột, index và thứ tự của `df`
  (không thêm cột `n_quan` vào kết quả hay vào `df`).

Đối chiếu snapshot: **16.850** phòng thuộc **8** quận có ≥ 300 phòng.

In [ ]:
def big_districts(df: pd.DataFrame, nguong: int = 300) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q6():
    result = big_districts(DEMO, 4)
    assert set(result) == {"n_quan", "df_dong"}
    n_quan = result["n_quan"]
    assert isinstance(n_quan, pd.Series) and len(n_quan) == len(DEMO)
    assert list(n_quan) == [5, 5, 5, 5, 5, 4, 4, 4, 4, 2, 2, 1]
    df_dong = result["df_dong"]
    assert list(df_dong.columns) == list(DEMO.columns) and len(df_dong) == 9
    assert df_dong["neighbourhood"].nunique() == 2 and "n_quan" not in DEMO.columns

public_check("Q6 · ví dụ công khai", check_q6)

In [ ]:
def thuc_hanh_q6():
    nguong = 300 if LISTINGS_PATH is not None else 4
    df_dong = big_districts(df, nguong)["df_dong"]
    return f"{len(df_dong):,} phòng thuộc {df_dong['neighbourhood'].nunique()} quận có ≥ {nguong} phòng."

preview("Bài 6 trên dữ liệu đang dùng", thuc_hanh_q6)

## Bài 7 · `pivot_table` — phân khúc × kiểu host

### Hợp đồng hàm · 10 điểm tự động

`segment_host_pivot(df: pd.DataFrame) -> pd.DataFrame`

Nhận bảng có cột `phan_khuc` (chuỗi hoặc thiếu), `chuyen` (bool), `number_of_reviews_ltm`.
Trả `pivot_table` với `index="phan_khuc"`, `columns="chuyen"`, `values="number_of_reviews_ltm"`, `aggfunc="mean"`.
Tổ hợp không có dòng nào để NaN (hành vi mặc định). Không làm tròn, không sửa input.
Đối chiếu snapshot: trung × chuyên nghiệp ≈ **16,5**; cao × cá nhân ≈ **9,5** (cao × chuyên nghiệp ≈ 13,5).

In [ ]:
def segment_host_pivot(df: pd.DataFrame) -> pd.DataFrame:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q7():
    demo = DEMO.assign(
        phan_khuc=["re", "trung", "cao", None, "trung", "trung", "trung", "cao", "re", "trung", "cao", "cao"],
        chuyen=DEMO["calculated_host_listings_count"] >= 5)
    pv = segment_host_pivot(demo)
    assert isinstance(pv, pd.DataFrame)
    assert set(pv.index) == {"re", "trung", "cao"} and set(pv.columns) == {False, True}
    assert pv.loc["cao", False] == 1.0 and round(pv.loc["trung", True], 2) == 21.67
    assert np.isnan(pv.loc["re", True]), "Phân khúc rẻ không có host chuyên nghiệp → NaN"

public_check("Q7 · ví dụ công khai", check_q7)

In [ ]:
preview("Bài 7 trên dữ liệu đang dùng", lambda: segment_host_pivot(df).round(1))

**Đọc bảng chéo:** ở phân khúc nào host chuyên nghiệp có review trung bình cao hơn? Kết quả Bài 5 còn đứng vững khi tách theo phân khúc không? Đây đã là bằng chứng nhân quả chưa? …

## Bài 8 · `merge` bảng tra cứu + `validate`

Giả sử công ty áp **phí dịch vụ theo phân khúc**: rẻ 10%, trung 13%, cao 15%.
Bảng tra cứu gồm 3 dòng và được ghép vào bảng listing (18.534 dòng trên snapshot).

### Hợp đồng hàm · 10 điểm tự động

`merge_fees(df: pd.DataFrame, phi: pd.DataFrame) -> dict`

Nhận bảng listing có cột `phan_khuc` (chuỗi hoặc thiếu) và bảng tra cứu `phi` có cột `phan_khuc`, `phi_dich_vu`.
`merge` theo `phan_khuc`, `how="left"`, **`validate="m:1"`**. Trả dict có đúng các khóa:
`m` (DataFrame sau merge), `so_dong_m` (int, số dòng của `m`), `so_nan_phi` (int, số NaN của cột `phi_dich_vu`).
Nếu `phi` có khóa trùng, **để lỗi `MergeError` của `validate` được ném ra** (không bắt lỗi, không tự bỏ trùng).
Không sửa input. Đối chiếu snapshot: `m` giữ **18.534** dòng; **846** NaN — đúng bằng số phòng không có giá.

In [ ]:
phi = pd.DataFrame({"phan_khuc": ["re", "trung", "cao"],
                    "phi_dich_vu": [0.10, 0.13, 0.15]})

In [ ]:
def merge_fees(df: pd.DataFrame, phi: pd.DataFrame) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q8():
    demo = DEMO.assign(phan_khuc=DEMO["price"].map(
        lambda g: None if pd.isna(g) else "re" if g < 30000 else "trung" if g < 100000 else "cao"))
    result = merge_fees(demo, phi)
    assert set(result) == {"m", "so_dong_m", "so_nan_phi"}
    assert result["so_dong_m"] == 12, "merge làm đổi số dòng?!"
    assert result["so_nan_phi"] == 1, "NaN đúng bằng số phòng không có phân khúc (không giá)"
    m = result["m"]
    assert list(m.columns) == list(demo.columns) + ["phi_dich_vu"]
    gia_tong_tv = (m["price"] * (1 + m["phi_dich_vu"])).groupby(m["phan_khuc"]).median().round(0)
    assert gia_tong_tv["trung"] == 62150.0
    trung_khoa = pd.concat([phi, phi.head(1)])
    try:
        merge_fees(demo, trung_khoa)
    except pd.errors.MergeError:
        pass
    else:
        raise AssertionError("validate='m:1' phải báo lỗi khi bảng tra cứu có khóa trùng")

public_check("Q8 · ví dụ công khai", check_q8)

In [ ]:
def thuc_hanh_q8():
    ket_qua = merge_fees(df, phi)
    m = ket_qua["m"]
    gia_tong = m["price"] * (1 + m["phi_dich_vu"])
    print(f"{ket_qua['so_dong_m']:,} dòng sau merge (trước merge: {len(df):,}); "
          f"{ket_qua['so_nan_phi']:,} NaN ở phi_dich_vu")
    return gia_tong.groupby(m["phan_khuc"]).median().round(0)

preview("Bài 8 trên dữ liệu đang dùng — trung vị giá gồm phí", thuc_hanh_q8)

**Giải thích NaN sau merge:** cột phí có … NaN vì …; `validate="m:1"` bảo vệ điều gì? …

## Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà) — không cộng vào 80 điểm tự động

Bạn được dùng AI theo quy trình 5 bước; hãy ghi lại prompt và cách kiểm chứng.

### Tự làm 1 · Phòng đắt nhất từng quận

Với các quận lớn (`df_dong` của Bài 6): dùng `transform("max")` trên giá theo quận để tìm **phòng đắt
nhất của từng quận** bằng cách so sánh `price` với giá lớn nhất của quận. In các cột
`id`, `neighbourhood` và `price` (snapshot có thêm cột `name`).
Kiểm chứng trên snapshot: giá lớn nhất của quận Santiago phải là 97.000.045 CLP.

### Tự làm 2 · Cơ cấu phân khúc theo quận

Dùng `pivot_table` (hoặc `groupby` + `value_counts(normalize=True)`) tính **tỷ trọng
3 phân khúc trong từng quận lớn**. Quận nào có tỷ trọng phân khúc cao lớn nhất? Kết quả có
khớp với bảng xếp hạng trung vị của lab 2 (Lo Barnechea, Vitacura dẫn đầu) không?

In [ ]:
# Viết bài tự làm của bạn ở đây (không được chấm tự động).

**Bài tự làm của tôi:** …

## Khai báo sử dụng AI

- **Công cụ đã dùng:** … *(hoặc “không dùng”)*
- **Bài đã dùng AI và yêu cầu chính:** …
- **Một gợi ý đã kiểm chứng, cách kiểm chứng và kết quả:** …

Không chỉ ghi “code chạy được”; nêu phép tính tay, số dòng hoặc ví dụ đối chiếu đã dùng.

---

## Tóm tắt bài lab

| Nội dung chính | Sẽ gặp lại ở |
|---|---|
| `loc` theo nhãn, `iloc` theo vị trí; alignment theo index | mọi phép toán giữa hai bảng/Series |
| `map` với hàm sinh cột phân loại | tạo biến phân loại từ dữ liệu số |
| `agg` đặt tên nhiều cột — đọc bảng thành nhận xét | xây dựng bảng tổng hợp và KPI |
| `transform` gắn thông tin nhóm về dòng (`size`, `max`) | kiểm tra ngoại lai theo nhóm (bài 10) |
| `pivot_table` kiểm tra kết luận theo biến phân nhóm khác | phân tích bảng chéo (bài 14) |
| `merge` + `validate` + giải thích từng NaN | ghép bảng vùng hoặc tỷ giá |

**Trước khi nộp:** hoàn thành TODO và câu trả lời, chạy Restart & Run all, lưu notebook.

**Bài giảng tiếp theo:** làm việc với dữ liệu từ API, SQL và DuckDB.

## Thông tin và nộp bài

**Họ tên / MSSV / lớp:** [Điền]

1. Giữ tên file **`lab-05.ipynb`** ở gốc repo (dấu gạch ngang, không phải `lab_05.ipynb`).
2. Hoàn thành hàm TODO, các dự đoán, diễn giải bảng và khai báo AI.
3. Restart & Run all; đọc bảng public checks, lưu file rồi commit/push lên `main`.
4. Tạo tag `submit/lab-05/v1`; lần sửa tiếp dùng v2 hoặc v3, không di chuyển tag đã nộp.
5. Repo cần được giảng viên đăng ký chấm. Xem status `private-grader/lab-05` trên commit nộp;
   điểm tự động tối đa 80, còn 20 điểm chờ giảng viên. Xem lịch quét trong README.

Tag v1/v2/v3 là **lần nộp**, không phải phiên bản đề. Không đổi metadata để chọn bộ test khác.

In [ ]:
show_public_summary()